# Create video with fly on ball, brain activity, and fly behavior 

## Imports

In [1]:
from pathlib import Path
import os.path
from datetime import datetime
import pickle
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import CenteredNorm 
import numpy as np
import warnings
import cv2
import pingouin


from unityvr.preproc import logproc as lp
from unityvr.analysis import posAnalysis
from gulp2p.preproc import utils as utils
from gulp2p.preproc import imaging
from gulp2p.preproc import rois
from gulp2p.preproc.tiff import Tiff

c:\Users\ahshenas\Anaconda3\envs\gulp2p\lib\site-packages\outdated\utils.py:14: OutdatedPackageWarning: The package pingouin is out of date. Your version is 0.5.3, the latest is 0.5.4.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(


## How do I create a video with python?

https://matplotlib.org/stable/users/explain/animations/animations.html


### Files:

In [2]:
# fictrac_video_path = Path(r"Z:\Live Fly Imaging data\fictrac\fictrac-dbg-20240103_102052.avi")
# fictrac_video_path = Path(r"Z:\Live Fly Imaging data\fictrac\fictrac-dbg-20240110_113229.avi")
fictrac_video_path = Path(r"Z:\Live Fly Imaging data\fictrac\fictrac-dbg-20240110_113229_copy.avi")
# fictrac_video_path = Path(r"Z:\Live Fly Imaging data\fictrac\fictrac-raw-20240103_111219.avi")
tiff_path = Path(r"Z:\2PImaging\Kerstin\MIMS\20231207\20231207_DR019xiGluSnFr_Fly1_00001.tif")
synced_bhv_img_path = Path(r"Z:\2PImaging\Kerstin\MIMS\20231207\20231207_DR019xiGluSnFr_Fly1_00001.p")


In [3]:
with open(synced_bhv_img_path, 'rb') as pkl:
    exptDat = pickle.load(pkl)
expDf = exptDat['expDf']

### Read Tiff:

In [4]:
# Load tiff
tiff = Tiff(tiff_path)
tiff.metadata
zaxis = 1
mip_stack = np.squeeze(np.max(tiff.stack, axis=zaxis))
print(tiff.metadata)

{'laser_power': 0.35, 'SizeC': 1, 'discard_fb_frames': True, 'flyback_time': 0.001, 'num_fb_frames': 1, 'SizeY': 256, 'SizeX': 128, 'frame_interval': 0.017176, 'frame_rate': 58.2206, 'volume_rate': 6.46896, 'zoom_factor': 2.9, 'pixel_bin_factor': 1, 'SizeZ': 9, 'SizeT': 2350, 'date': datetime.datetime(2023, 12, 7, 9, 52, 39, 385401), 'file_size': 1436527630, 'dimension_order': 'TZCYX', 'pixel_width': 2.11512227715946, 'width_unit': 'um', 'pixel_height': 2.11512227715946, 'height_unit': 'um'}


## Plot using matplotlib animations

### Read Video:

In [5]:
def get_video_metadata(cap):
    metadata = {}
    # https://docs.opencv.org/4.x/d4/d15/group__videoio__flags__base.html
    # https://docs.opencv.org/4.x/d4/d15/group__videoio__flags__base.html#ggaeb8dd9c89c10a5c63c139bf7c4f5704da7c2fa550ba270713fca1405397b90ae0
    metadata['width']  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    metadata['height'] = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    metadata['frame_count'] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    metadata['fps'] = cap.get(cv2.CAP_PROP_FPS)
    metadata['bitrate'] = cap.get(cv2.CAP_PROP_BITRATE)
    if cap.get(cv2.CAP_PROP_CONVERT_RGB) == 1:
        metadata['channels'] = 3
    return metadata

def read_video_cv2(video_path):
    cap = cv2.VideoCapture(video_path)
    metadata = get_video_metadata(cap)
    shape = (metadata['frame_count'], metadata['height'], metadata['width'], metadata['channels'])
    # shape = (metadata['frame_count'], metadata['height'], metadata['width'])
    video = np.empty(shape=shape)
    i=0
    while cap.isOpened() and i<metadata['frame_count']:
        ret, frame = cap.read()
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        video[i,:] = frame
        i+=1
    cap.release()
    return video.astype(int), metadata

### Video functions:

In [6]:
def crop_frame(video_frame, xlim=[None]*2, ylim=[None]*2):
    frame = video_frame.copy()
    x_dim = 1
    y_dim = 0
    # crop the frame
    if xlim[0] is None:
        xlim[0] = 0
    if xlim[1] is None:
        xlim[1] = frame.shape[x_dim]
    x_slice = slice(int(xlim[0]), int(xlim[1]), None)
    
    if ylim[0] is None:
        ylim[0] = 0
    if ylim[1] is None:
        ylim[1] = frame.shape[y_dim]
    y_slice = slice(int(ylim[0]), int(ylim[1]), None)
    return frame[y_slice, x_slice]

### Create video:

In [7]:
# # mosaic = [['flor', 'movie'],
# #           ['bhv', 'bhv']]
# mosaic = [['flor', 'movie']]
# height_ratios=[1]
# fig, axd = plt.subplot_mosaic(mosaic=mosaic, height_ratios=height_ratios)

# # Read video
# cap = cv2.VideoCapture(fictrac_video_path.as_posix())
# video_metadata = get_video_metadata(cap)
# print(video_metadata)


# # Initial plot
# flor = axd['flor'].imshow(mip_stack[0], interpolation='none')
# # bhv = axd['bhv'].plot(expDf['vRfilt'][0:2])[0]
# ret, image = cap.read()
# crop_x_amount = 0.5
# crop_x_point = crop_x_amount * image.shape[1]
# image = crop_frame(image, xlim=[None, crop_x_point])
# mov = axd['movie'].imshow(image)

# axd['flor'].set_axis_off()
# axd['movie'].set_axis_off()
# # axd['flor'].set_xlim([0, exptDat['DF_G'].shape[0]])
# # axd['bhv'].set_xlim([0, len(expDf['vRfilt'])])

# plot_fps = 60
# def update(plot_frame):
#   artists = []

#   plot_time = plot_frame / plot_fps

#   # Step to current video frame
#   while True:
#     ret, image = cap.read()
#     # Don't update if image not returned
#     if not ret:
#       break
#     # Frame Info
#     image_time = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000 # Converts from ms to sec
#     image_frame = int(cap.get(cv2.CAP_PROP_POS_FRAMES))

#     # print(image_time)
#     # Exit loop once image frame matches the plot_frame
#     if image_time >= plot_time:
#       break
#     if image_frame >= video_metadata['frame_count']:
#       break

#   # Update video frame
#   if ret:
#     image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB).astype(int)

#     # Crop frame
#     crop_x_amount = 0.5
#     crop_x_point = crop_x_amount * image.shape[1]
#     image = crop_frame(image, xlim=[None, crop_x_point])

#     mov.set_data(image)
#     artists += [mov]

#   # Update tiff frame
#   # Get tiff frame for current plot time
#   tiff_frame = round(plot_time * tiff.metadata['volume_rate'])
#   if tiff_frame < mip_stack.shape[0]:
#     flor.set_data(mip_stack[tiff_frame])
#     artists += [flor]


#   # for each frame, update the data stored on each artist.
#   # Convert the current time into the next frame of expDf data
#   # expt_frame = int(expDf[expDf['posTime'] >= plot_frame_time]['frame'].iloc[0])
#   # # print(expt_frame)
#   # if expt_frame > len(expDf['vRfilt']):
#   #   expt_frame = len(expDf['vRfilt'])
#   # bhv.set_xdata(range(expt_frame))
#   # bhv.set_ydata(expDf['vRfilt'][:expt_frame])
#   # # axd['flor'].imshow(exptDat['DF_G'].T[:,:expt_frame], aspect=64, interpolation='none')
#   # flor.set_data(exptDat['DF_G'].T[:,:expt_frame])
#   # artists += [bhv, flor]

#   return artists

# tiff_length =  tiff.metadata['SizeT']/tiff.metadata['volume_rate']
# vid_length = video_metadata['frame_count'] / video_metadata['fps']
# plot_length = min(tiff_length, vid_length)

# plot_total_frames = int(plot_length * plot_fps) - 1
# fm_interval_ms = 1/plot_fps*1000

# ani = animation.FuncAnimation(fig=fig,
#                               func=update,
#                               # frames=60*1,
#                               frames=int(tiff_length*plot_fps),
#                               # frames=plot_total_frames,
#                               interval=fm_interval_ms,
#                               blit=True,
#                               )
# ani.save("../results/videos/animation_test.mp4",writer='ffmpeg') # Need to install ffmpeg (conda install -c conda-forge ffmpeg) 

# cap.release()

## Plot using MoviePy

### New imports

In [8]:
from moviepy.editor import VideoClip, VideoFileClip, clips_array, CompositeVideoClip, TextClip
from moviepy.video.io.bindings import mplfig_to_npimage

### Example code

In [9]:
def grayscale_to_rgb_frame(frame):
    # Reshape frame to include color then duplicate
    # rgb_frame = np.copy(frame)
    # rgb_frame = np.repeat(np.expand_dims(rgb_frame, axis=-1),
    #                       repeats=3, axis=-1)
    cm = plt.get_cmap('viridis')
    rgb_frame = cm(frame)
    # Cut off alpha channel
    rgb_frame = rgb_frame[:,:,:3] * 255
    return rgb_frame.astype(int)

In [10]:
def get_mip_stack(tiff):
    zaxis = 1
    mip_stack = np.squeeze(np.max(tiff.stack, axis=zaxis))
    return mip_stack

def get_frame_at_time(tiff, time, convert_to_rbg=False):
    tiff_frame_idx = round(time * tiff.metadata['volume_rate'])
    tiff_vol_count = tiff.metadata['SizeT']
    if tiff_frame_idx > tiff_vol_count:
        tiff_frame_idx = tiff_vol_count - 1
    mip_stack = get_mip_stack(tiff)
    
    frame = mip_stack[tiff_frame_idx]
    if convert_to_rbg:
        frame = grayscale_to_rgb_frame(frame)
    return frame

In [11]:
# def make_frame_matplotlib(time, ax):
#     ax.clear()
#     frame = get_frame_at_time(tiff, time, convert_to_rbg=False)
#     ax.imshow(frame, interpolation='none', cmap='viridis')
#     ax.set_axis_off()
#     return mplfig_to_npimage(fig)

In [12]:
def make_tiff_frame(time):
    # Given a timepoint return the frame at that time.
    frame = get_frame_at_time(tiff, time, convert_to_rbg=True)
    return frame

In [13]:
# # Display a few frames from the video
# length = 2
# num_frames = 5
# fig, axs = plt.subplots(ncols=5, figsize=(10,5))
# for index, time in enumerate(np.linspace(0,length, num_frames)):
#     frame = get_frame_at_time(tiff, time, convert_to_rbg=True)
#     # print(frame.shape)
#     axs[index].imshow(frame)
#     axs[index].set_axis_off()

In [14]:
duration = 10
fps = 60

In [15]:
fictrac_animation = (VideoFileClip(fictrac_video_path.as_posix())
                     .subclip(0,30)
                     .crop(x1=0, width=318)
                     .margin(left=10))

In [23]:
tiff_animation = (VideoClip(make_tiff_frame)
                  .resize(height=fictrac_animation.size[1]))
txt_clip = TextClip(txt = tiff.path.name, fontsize = 32, color = 'white')
txt_clip = txt_clip.set_pos('bottom').set_duration(duration)  

tiff_animation = CompositeVideoClip([tiff_animation, txt_clip])

In [24]:
# output = Path("../results/videos/moviepy_tiff_test.mp4").as_posix()
# tiff_animation.write_videofile(output, fps=tiff.metadata['volume_rate'])

### Assemble list of clips

In [25]:
# final_clip = clips_array([tiff_animation, fictrac_animation],
#                          bg_color=[255,255,255])

composite_video = CompositeVideoClip([tiff_animation.set_position(('left')),
                                      fictrac_animation.set_position(('right'))],
                                      size=(tiff_animation.size[0]+ fictrac_animation.size[0],
                                            fictrac_animation.size[1]))
composite_video = composite_video.subclip(0, duration)

In [30]:
composite_video.show(2)

ImportError: clip.show requires Pygame installed

In [26]:
output = Path("../results/videos/moviepy_combined_test.mp4").as_posix()
composite_video.write_videofile(output, fps=fps)

Moviepy - Building video ../results/videos/moviepy_combined_test.mp4.
Moviepy - Writing video ../results/videos/moviepy_combined_test.mp4



Moviepy - Done !
Moviepy - video ready ../results/videos/moviepy_combined_test.mp4


In [27]:
composite_video.close()

In [28]:
tiff_animation.size

(240, 480)

In [29]:
fictrac_animation.size

(328, 480)